# Weekly analytics — publish + agency pipeline

Scheduled: `scripts/run-notebook.sh weekly-analytics.ipynb` (host cron).

In [ ]:
import sys, os, datetime
sys.path.insert(0, '/home/jovyan/work')
from _lib import db, slack

week = datetime.datetime.utcnow() - datetime.timedelta(days=7)

pub = db("SELECT status, count(*) FROM publish_queue "
         f"WHERE created_at >= '{week:%Y-%m-%d}' GROUP BY 1")

agency = db("SELECT status, count(*) FROM agency_proposals "
            f"WHERE created_at >= '{week:%Y-%m-%d}' GROUP BY 1")

audits = db("SELECT avg(score)::numeric(5,1) AS avg_score, count(*) "
            f"FROM agency_audits WHERE created_at >= '{week:%Y-%m-%d}'")

msg = (f'*Weekly SocialAuto — {datetime.date.today()}*\n'
       f'Publish queue last 7d: `{pub.to_dict("records")}`\n'
       f'Agency proposals: `{agency.to_dict("records")}`\n'
       f'Audits run: `{audits.iloc[0].to_dict()}`')
print(msg)
ok = slack(msg)
print('slack:', ok)
